# Insights Avançados — Exploração Independente da Gold Layer

**Objetivo:** Ir além da análise comercial padrão e revelar padrões, anomalias, oportunidades e riscos não-óbvios nos dados de saúde suplementar.

**Catálogo/Schema:** `homologacao.salutar_saude`  
**Período de análise:** Jul/2024 – Jun/2025

---

## Índice de Achados

1. **Sazonalidade de Sinistros** — Existe um "calendário de custo"?
2. **Seleção Adversa** — Beneficiários novos geram sinistros desproporcionais nos primeiros meses?
3. **Qualidade da Venda por Corretor** — Quem vende contratos que geram prejuízo?
4. **Concentração de Receita** — Quão vulnerável é a carteira? (Índice HHI)
5. **Escalação de Eventos** — PS leva a internação/cirurgia no mesmo beneficiário?
6. **Eficiência de Rede por Operadora** — Custos sistemáticos diferentes entre operadoras?
7. **Outliers Estatísticos** — Eventos com custo anômalo que merecem auditoria

## 1. Sazonalidade de Sinistros — O "Calendário de Custo"

**Pergunta:** Existem meses sistematicamente mais caros? Certos tipos de evento ou especialidades têm padrão sazonal?

**Por que importa:** Se há picos previsíveis, o time pode provisionar reservas técnicas e negociar antecipadamente com a rede credenciada.

**O que fazer:** Alinhar provisionamento de IBNR com a curva sazonal; antecipar negociação com hospitais antes dos meses de pico.

In [0]:
%sql
-- Sazonalidade: custo mensal por tipo de evento
SELECT
  DATE_TRUNC('MONTH', data_evento) AS mes,
  tipo_evento,
  COUNT(*) AS qtd_eventos,
  ROUND(SUM(valor_sinistro), 2) AS custo_total,
  ROUND(AVG(valor_sinistro), 2) AS ticket_medio
FROM homologacao.salutar_saude.gold_fat_utilizacao
WHERE data_evento BETWEEN DATE '2024-07-01' AND DATE '2025-06-30'
GROUP BY DATE_TRUNC('MONTH', data_evento), tipo_evento
ORDER BY mes, custo_total DESC

mes,tipo_evento,qtd_eventos,custo_total,ticket_medio
2024-07-01T00:00:00.000Z,Cirurgia,35,2567478.54,73356.53
2024-07-01T00:00:00.000Z,Internação,41,1548568.64,37769.97
2024-07-01T00:00:00.000Z,Exame De Imagem,40,66541.80,1663.55
2024-07-01T00:00:00.000Z,Pronto-socorro,23,31299.89,1360.86
2024-07-01T00:00:00.000Z,Exame,27,17353.03,642.70
2024-07-01T00:00:00.000Z,Consulta,38,9886.86,260.18
2024-07-01T00:00:00.000Z,Terapia,34,7366.75,216.67
2024-08-01T00:00:00.000Z,Cirurgia,36,2779329.21,77203.59
2024-08-01T00:00:00.000Z,Internação,28,1159400.06,41407.15
2024-08-01T00:00:00.000Z,Exame De Imagem,32,62659.19,1958.10


In [0]:
%sql
-- Detectar meses anômalos: custo de cirurgia vs. média móvel
WITH mensal AS (
  SELECT
    DATE_TRUNC('MONTH', data_evento) AS mes,
    COUNT(*) AS qtd_cirurgias,
    ROUND(SUM(valor_sinistro), 2) AS custo_cirurgias
  FROM homologacao.salutar_saude.gold_fat_utilizacao
  WHERE data_evento BETWEEN DATE '2024-07-01' AND DATE '2025-06-30'
    AND tipo_evento = 'Cirurgia'
  GROUP BY DATE_TRUNC('MONTH', data_evento)
)
SELECT
  mes,
  qtd_cirurgias,
  custo_cirurgias,
  ROUND(AVG(custo_cirurgias) OVER (ORDER BY mes ROWS BETWEEN 2 PRECEDING AND 2 FOLLOWING), 2) AS media_movel_5m,
  ROUND((custo_cirurgias - AVG(custo_cirurgias) OVER ()) / STDDEV(custo_cirurgias) OVER (), 2) AS z_score,
  CASE
    WHEN (custo_cirurgias - AVG(custo_cirurgias) OVER ()) / STDDEV(custo_cirurgias) OVER () > 1.5 THEN '🔴 PICO'
    WHEN (custo_cirurgias - AVG(custo_cirurgias) OVER ()) / STDDEV(custo_cirurgias) OVER () < -1.5 THEN '🟢 VALE'
    ELSE '⚪ NORMAL'
  END AS alerta
FROM mensal
ORDER BY mes

mes,qtd_cirurgias,custo_cirurgias,media_movel_5m,z_score,alerta
2024-07-01T00:00:00.000Z,35,2567478.54,3104538.90,-1.17,⚪ NORMAL
2024-08-01T00:00:00.000Z,36,2779329.21,3542535.07,-1.08,⚪ NORMAL
2024-09-01T00:00:00.000Z,45,3966808.94,3717658.60,-0.56,⚪ NORMAL
2024-10-01T00:00:00.000Z,61,4856523.60,4017116.26,-0.17,⚪ NORMAL
2024-11-01T00:00:00.000Z,54,4418152.70,4295752.13,-0.36,⚪ NORMAL
2024-12-01T00:00:00.000Z,54,4064766.84,4407154.70,-0.52,⚪ NORMAL
2025-01-01T00:00:00.000Z,54,4172508.59,4761457.45,-0.47,⚪ NORMAL
2025-02-01T00:00:00.000Z,63,4523821.76,5145677.73,-0.32,⚪ NORMAL
2025-03-01T00:00:00.000Z,84,6628037.37,5982421.95,0.61,⚪ NORMAL
2025-04-01T00:00:00.000Z,82,6339254.07,7220005.09,0.48,⚪ NORMAL


Databricks visualization. Run in Databricks to view.

## 2. Seleção Adversa — Beneficiários Novos São Mais Caros?

**Pergunta:** Beneficiários recém-aderidos geram custo desproporcional nos primeiros meses? Isso indicaria seleção adversa (pessoas entrando no plano já doentes).

**Por que importa:** Se confirmado, a política de carência ou a subscripção médica pode estar falhando. Impacto direto na precificação.

**O que fazer:** Reforçar triagem na adesão para grupos de risco; considerar carência progressiva para procedimentos de alto custo.

In [0]:
%sql
-- Seleção adversa: custo médio por "idade do beneficiário no plano"
-- Meses entre data_adesao e data_evento
WITH eventos AS (
  SELECT
    u.beneficiario_id,
    u.valor_sinistro,
    u.tipo_evento,
    u.data_evento,
    b.data_adesao,
    MONTHS_BETWEEN(u.data_evento, b.data_adesao) AS meses_no_plano
  FROM homologacao.salutar_saude.gold_fat_utilizacao u
    JOIN homologacao.salutar_saude.gold_dim_beneficiario b ON u.beneficiario_id = b.beneficiario_id
  WHERE u.data_evento BETWEEN DATE '2024-07-01' AND DATE '2025-06-30'
    AND b.data_adesao IS NOT NULL
)
SELECT
  CASE
    WHEN meses_no_plano < 3 THEN '00-02 meses'
    WHEN meses_no_plano < 6 THEN '03-05 meses'
    WHEN meses_no_plano < 12 THEN '06-11 meses'
    WHEN meses_no_plano < 24 THEN '12-23 meses'
    ELSE '24+ meses'
  END AS tenure_grupo,
  COUNT(*) AS qtd_eventos,
  COUNT(DISTINCT beneficiario_id) AS benef_distintos,
  ROUND(SUM(valor_sinistro), 2) AS custo_total,
  ROUND(AVG(valor_sinistro), 2) AS ticket_medio,
  ROUND(SUM(valor_sinistro) / COUNT(DISTINCT beneficiario_id), 2) AS custo_per_capita,
  ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT beneficiario_id), 1) AS freq_per_capita,
  -- % que é cirurgia
  ROUND(SUM(CASE WHEN tipo_evento = 'Cirurgia' THEN valor_sinistro ELSE 0 END) * 100.0 / SUM(valor_sinistro), 1) AS pct_custo_cirurgia
FROM eventos
GROUP BY
  CASE
    WHEN meses_no_plano < 3 THEN '00-02 meses'
    WHEN meses_no_plano < 6 THEN '03-05 meses'
    WHEN meses_no_plano < 12 THEN '06-11 meses'
    WHEN meses_no_plano < 24 THEN '12-23 meses'
    ELSE '24+ meses'
  END
ORDER BY tenure_grupo

tenure_grupo,qtd_eventos,benef_distintos,custo_total,ticket_medio,custo_per_capita,freq_per_capita,pct_custo_cirurgia
00-02 meses,1699,588,32902981.55,19366.09,55957.45,2.9,65.0
03-05 meses,1003,533,20086840.45,20026.76,37686.38,1.9,67.2
06-11 meses,1408,675,23965483.90,17020.94,35504.42,2.1,61.0
12-23 meses,1127,473,20959482.45,18597.59,44311.80,2.4,63.9
24+ meses,27,20,336886.05,12477.26,16844.30,1.4,9.5


Databricks visualization. Run in Databricks to view.

## 3. Qualidade da Venda por Corretor — Quem Vende Prejuízo?

**Pergunta:** Correlacionando o corretor da venda com a sinistralidade subsequente do contrato vendido, há corretores que sistematicamente trazem carteiras de risco?

**Por que importa:** Corretor com alta produção mas alta sinistralidade não é bom vendedor — está gerando receita que vira prejuízo. Pode indicar falta de subscripção adequada ou perfil de venda orientado apenas a comissão.

**O que fazer:** Criar score de qualidade de venda (receita × sinistralidade); ajustar comissionamento para penalizar sinistralidade elevada após 12 meses.

In [0]:
%sql
-- Qualidade da venda: corretor × sinistralidade dos contratos vendidos
WITH corretor_receita AS (
  SELECT
    c.corretor_nome,
    c.regiao,
    c.senioridade,
    COUNT(f.contrato_id) AS contratos_vendidos,
    SUM(f.num_vidas) AS vidas_vendidas,
    ROUND(SUM(f.receita_total_mensal), 2) AS receita_gerada
  FROM homologacao.salutar_saude.gold_fat_contratos f
    JOIN homologacao.salutar_saude.gold_dim_corretor c ON f.corretor_id = c.corretor_id
  GROUP BY c.corretor_nome, c.regiao, c.senioridade
),
corretor_sinistro AS (
  SELECT
    fc.corretor_id,
    ROUND(SUM(u.valor_sinistro), 2) AS custo_sinistros
  FROM homologacao.salutar_saude.gold_fat_utilizacao u
    JOIN homologacao.salutar_saude.gold_fat_contratos fc ON u.contrato_id = fc.contrato_id
  WHERE u.data_evento BETWEEN DATE '2024-07-01' AND DATE '2025-06-30'
  GROUP BY fc.corretor_id
)
SELECT
  cr.corretor_nome,
  cr.regiao,
  cr.senioridade,
  cr.contratos_vendidos,
  cr.vidas_vendidas,
  cr.receita_gerada,
  COALESCE(cs.custo_sinistros, 0) AS custo_sinistros_12m,
  ROUND(COALESCE(cs.custo_sinistros, 0) * 100.0 / NULLIF(cr.receita_gerada, 0), 1) AS sinistralidade_corretor_pct,
  ROUND(cr.receita_gerada - COALESCE(cs.custo_sinistros, 0), 2) AS margem_liquida,
  -- Score: margem positiva = bom, negativa = ruim
  CASE
    WHEN COALESCE(cs.custo_sinistros, 0) * 100.0 / NULLIF(cr.receita_gerada, 0) > 100 THEN '🔴 VENDA COM PREJUÍZO'
    WHEN COALESCE(cs.custo_sinistros, 0) * 100.0 / NULLIF(cr.receita_gerada, 0) > 75 THEN '🟡 ATENÇÃO'
    ELSE '🟢 SAUDÁVEL'
  END AS score_qualidade
FROM corretor_receita cr
  LEFT JOIN homologacao.salutar_saude.gold_dim_corretor dc ON cr.corretor_nome = dc.corretor_nome
  LEFT JOIN corretor_sinistro cs ON dc.corretor_id = cs.corretor_id
ORDER BY sinistralidade_corretor_pct DESC

corretor_nome,regiao,senioridade,contratos_vendidos,vidas_vendidas,receita_gerada,custo_sinistros_12m,sinistralidade_corretor_pct,margem_liquida,score_qualidade
Tiago Nunes,Norte,Pleno,3,610,140455849.10,4246697.07,3.0,136209152.03,🟢 SAUDÁVEL
Igor Oliveira,Nordeste,Júnior,6,1654,328138531.35,7816567.36,2.4,320321963.99,🟢 SAUDÁVEL
Zeca Almeida,Centro-Oeste,Júnior,2,405,50293557.96,1179773.49,2.3,49113784.47,🟢 SAUDÁVEL
Paulo Santos,Centro-Oeste,Júnior,4,815,107513767.88,2135829.13,2.0,105377938.75,🟢 SAUDÁVEL
Eduardo Rodrigues,Nordeste,Pleno,4,1098,235071049.42,4222971.85,1.8,230848077.57,🟢 SAUDÁVEL
Gabriel Silva,Norte,Júnior,3,1158,259942562.08,4643723.55,1.8,255298838.53,🟢 SAUDÁVEL
Tiago Rocha,Sudeste,Sênior,2,779,232345348.43,2707154.68,1.2,229638193.75,🟢 SAUDÁVEL
Ursula Pereira,Nordeste,Júnior,2,422,53752326.18,635947.69,1.2,53116378.49,🟢 SAUDÁVEL
Nathan Rocha,Norte,Pleno,3,1078,203394774.76,2279954.42,1.1,201114820.34,🟢 SAUDÁVEL
Diego Oliveira,Sudeste,Júnior,7,2527,898016059.48,9688339.37,1.1,888327720.11,🟢 SAUDÁVEL


## 4. Concentração de Receita — Vulnerabilidade da Carteira

**Pergunta:** Quantos contratos representam a maior parte da receita? Se um único contrato grande cancelar, qual o impacto?

**Por que importa:** Alta concentração = alto risco de perda súbita de receita. O Índice HHI (Herfindahl-Hirschman) mede essa vulnerabilidade: >2500 = altamente concentrado.

**O que fazer:** Diversificar a carteira; mapear os top 5 contratos e criar plano de retenção premium; estabelecer meta de novos contratos para diluir concentração.

In [0]:
%sql
-- Concentração de receita: HHI + análise de dependência
WITH receita_contrato AS (
  SELECT
    f.contrato_id,
    e.empresa_nome,
    f.receita_total_mensal,
    f.receita_total_mensal * 100.0 / SUM(f.receita_total_mensal) OVER () AS pct_receita
  FROM homologacao.salutar_saude.gold_fat_contratos f
    JOIN homologacao.salutar_saude.gold_dim_empresa e ON f.empresa_id = e.empresa_id
  WHERE f.status = 'Ativo'
),
hhi AS (
  SELECT ROUND(SUM(pct_receita * pct_receita), 1) AS indice_hhi
  FROM receita_contrato
)
SELECT
  rc.contrato_id,
  rc.empresa_nome,
  ROUND(rc.receita_total_mensal, 2) AS receita_mensal,
  ROUND(rc.pct_receita, 2) AS pct_receita_total,
  ROUND(SUM(rc.pct_receita) OVER (ORDER BY rc.receita_total_mensal DESC), 2) AS pct_acumulado,
  ROW_NUMBER() OVER (ORDER BY rc.receita_total_mensal DESC) AS rank,
  hhi.indice_hhi AS hhi_carteira,
  CASE
    WHEN hhi.indice_hhi > 2500 THEN '🔴 MUITO CONCENTRADA'
    WHEN hhi.indice_hhi > 1500 THEN '🟡 MODERADA'
    ELSE '🟢 DIVERSIFICADA'
  END AS classificacao_hhi
FROM receita_contrato rc
  CROSS JOIN hhi
ORDER BY rc.receita_total_mensal DESC
LIMIT 20

contrato_id,empresa_nome,receita_mensal,pct_receita_total,pct_acumulado,rank,hhi_carteira,classificacao_hhi
44,Empresa Q17 Ltda,404725526.40,9.13,9.13,1,414.4,🟢 DIVERSIFICADA
9,Empresa D04 Ltda,367393773.60,8.29,17.43,2,414.4,🟢 DIVERSIFICADA
82,Empresa D30 Ltda,227552739.75,5.14,22.56,3,414.4,🟢 DIVERSIFICADA
68,Empresa Z26 Ltda,223002213.76,5.03,27.59,4,414.4,🟢 DIVERSIFICADA
75,Empresa B28 Ltda,219200707.91,4.95,32.54,5,414.4,🟢 DIVERSIFICADA
42,Empresa Q17 Ltda,210610699.92,4.75,37.29,6,414.4,🟢 DIVERSIFICADA
58,Empresa V22 Ltda,206066474.25,4.65,41.94,7,414.4,🟢 DIVERSIFICADA
93,Empresa G33 Ltda,189859327.04,4.28,46.23,8,414.4,🟢 DIVERSIFICADA
87,Empresa F32 Ltda,167166748.56,3.77,50.00,9,414.4,🟢 DIVERSIFICADA
59,Empresa W23 Ltda,166047470.82,3.75,53.75,10,414.4,🟢 DIVERSIFICADA


Databricks visualization. Run in Databricks to view.

## 5. Escalação de Eventos — PS → Internação → Cirurgia?

**Pergunta:** Beneficiários que utilizam pronto-socorro têm maior probabilidade de gerar internações ou cirurgias nos 90 dias seguintes? Existe um "funil de custos"?

**Por que importa:** Se PS funciona como precursor de eventos caros, investir em atenção primária/ambulatorial para interceptar esse funil pode ser a alavanca de maior ROI.

**O que fazer:** Criar programa de follow-up pós-PS (ligação em 48h, consulta agendada em 7 dias); medir taxa de escalação como indicador de qualidade assistencial.

In [0]:
%sql
-- Escalação: beneficiários com PS que tiveram cirurgia/internação em até 90 dias
WITH ps_events AS (
  SELECT DISTINCT
    beneficiario_id,
    data_evento AS data_ps
  FROM homologacao.salutar_saude.gold_fat_utilizacao
  WHERE tipo_evento = 'Pronto-socorro'
    AND data_evento BETWEEN DATE '2024-07-01' AND DATE '2025-03-31' -- janela para observar 90 dias
),
escalacao AS (
  SELECT
    ps.beneficiario_id,
    ps.data_ps,
    u.tipo_evento AS evento_subsequente,
    u.data_evento AS data_subsequente,
    u.valor_sinistro,
    DATEDIFF(u.data_evento, ps.data_ps) AS dias_ate_escalacao
  FROM ps_events ps
    JOIN homologacao.salutar_saude.gold_fat_utilizacao u
      ON ps.beneficiario_id = u.beneficiario_id
      AND u.data_evento > ps.data_ps
      AND u.data_evento <= DATE_ADD(ps.data_ps, 90)
      AND u.tipo_evento IN ('Cirurgia', 'Internação')
)
SELECT
  'Beneficiários com PS no período' AS metrica,
  (SELECT COUNT(DISTINCT beneficiario_id) FROM ps_events) AS valor
UNION ALL
SELECT
  'Desses, escalaram para Cirurgia/Internação em 90d',
  COUNT(DISTINCT beneficiario_id)
FROM escalacao
UNION ALL
SELECT
  'Taxa de escalação (%)',
  ROUND(COUNT(DISTINCT beneficiario_id) * 100.0 / (SELECT COUNT(DISTINCT beneficiario_id) FROM ps_events), 1)
FROM escalacao
UNION ALL
SELECT
  'Custo total dos eventos escalados (R$)',
  ROUND(SUM(valor_sinistro), 0)
FROM escalacao
UNION ALL
SELECT
  'Dias médios até escalação',
  ROUND(AVG(dias_ate_escalacao), 0)
FROM escalacao

metrica,valor
Beneficiários com PS no período,360.0
"Desses, escalaram para Cirurgia/Internação em 90d",104.0
Taxa de escalação (%),28.9
Custo total dos eventos escalados (R$),1.1108405E7
Dias médios até escalação,48.0


In [0]:
%sql
-- Funil PS → Cirurgia/Internação detalhado por especialidade do PS original
-- Qual especialidade do PS tem maior taxa de escalação?
WITH ps_events AS (
  SELECT
    beneficiario_id,
    evento_id AS ps_evento_id,
    especialidade AS especialidade_ps,
    data_evento AS data_ps,
    valor_sinistro AS custo_ps
  FROM homologacao.salutar_saude.gold_fat_utilizacao
  WHERE tipo_evento = 'Pronto-socorro'
    AND data_evento BETWEEN DATE '2024-07-01' AND DATE '2025-03-31'
),
escalacao AS (
  SELECT
    ps.beneficiario_id,
    ps.especialidade_ps,
    ps.data_ps,
    u.tipo_evento AS evento_escalado,
    u.especialidade AS especialidade_escalada,
    u.valor_sinistro AS custo_escalado,
    DATEDIFF(u.data_evento, ps.data_ps) AS dias_ate_escalacao
  FROM ps_events ps
    JOIN homologacao.salutar_saude.gold_fat_utilizacao u
      ON ps.beneficiario_id = u.beneficiario_id
      AND u.data_evento > ps.data_ps
      AND u.data_evento <= DATE_ADD(ps.data_ps, 90)
      AND u.tipo_evento IN ('Cirurgia', 'Internação')
),
resumo_ps AS (
  SELECT
    especialidade_ps,
    COUNT(DISTINCT beneficiario_id) AS benef_com_ps
  FROM ps_events
  GROUP BY especialidade_ps
)
SELECT
  r.especialidade_ps,
  r.benef_com_ps,
  COUNT(DISTINCT e.beneficiario_id) AS benef_escalaram,
  ROUND(COUNT(DISTINCT e.beneficiario_id) * 100.0 / r.benef_com_ps, 1) AS taxa_escalacao_pct,
  COUNT(*) AS total_eventos_escalados,
  ROUND(SUM(e.custo_escalado), 2) AS custo_total_escalado,
  ROUND(AVG(e.custo_escalado), 2) AS ticket_medio_escalado,
  ROUND(AVG(e.dias_ate_escalacao), 0) AS dias_medios_ate_escalacao,
  -- Mix do que escala: % cirurgia vs internação
  ROUND(SUM(CASE WHEN e.evento_escalado = 'Cirurgia' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_escala_cirurgia,
  ROUND(SUM(CASE WHEN e.evento_escalado = 'Internação' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_escala_internacao
FROM resumo_ps r
  LEFT JOIN escalacao e ON r.especialidade_ps = e.especialidade_ps
GROUP BY r.especialidade_ps, r.benef_com_ps
ORDER BY taxa_escalacao_pct DESC

especialidade_ps,benef_com_ps,benef_escalaram,taxa_escalacao_pct,total_eventos_escalados,custo_total_escalado,ticket_medio_escalado,dias_medios_ate_escalacao,pct_escala_cirurgia,pct_escala_internacao
Ortopedia,33,16,48.5,20,1301673.56,65083.68,41.0,50.0,50.0
Pediatria,50,18,36.0,25,1510807.46,60432.30,49.0,48.0,52.0
Cardiologia,35,12,34.3,17,1117104.44,65712.03,56.0,47.1,52.9
Psiquiatria,42,13,31.0,24,1379008.90,57458.70,50.0,37.5,62.5
Oftalmologia,37,10,27.0,22,1031316.69,46878.03,41.0,40.9,59.1
Endocrinologia,39,10,25.6,11,739478.90,67225.35,55.0,63.6,36.4
Dermatologia,37,9,24.3,13,663754.98,51058.08,38.0,53.8,46.2
Ginecologia,33,8,24.2,10,579696.96,57969.70,47.0,60.0,40.0
Clínica Geral,46,10,21.7,22,1530992.07,69590.55,43.0,77.3,22.7
Oncologia,68,14,20.6,25,1254571.51,50182.86,55.0,40.0,60.0


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Matriz de transição: de qual especialidade no PS para qual especialidade na cirurgia/internação?
WITH ps_events AS (
  SELECT
    beneficiario_id,
    especialidade AS especialidade_ps,
    data_evento AS data_ps
  FROM homologacao.salutar_saude.gold_fat_utilizacao
  WHERE tipo_evento = 'Pronto-socorro'
    AND data_evento BETWEEN DATE '2024-07-01' AND DATE '2025-03-31'
),
escalacao AS (
  SELECT
    ps.especialidade_ps,
    u.especialidade AS especialidade_destino,
    u.tipo_evento AS tipo_destino,
    u.valor_sinistro,
    DATEDIFF(u.data_evento, ps.data_ps) AS dias
  FROM ps_events ps
    JOIN homologacao.salutar_saude.gold_fat_utilizacao u
      ON ps.beneficiario_id = u.beneficiario_id
      AND u.data_evento > ps.data_ps
      AND u.data_evento <= DATE_ADD(ps.data_ps, 90)
      AND u.tipo_evento IN ('Cirurgia', 'Internação')
)
SELECT
  especialidade_ps AS ps_especialidade_origem,
  especialidade_destino AS destino_especialidade,
  tipo_destino AS tipo_evento_destino,
  COUNT(*) AS qtd_escalacoes,
  ROUND(SUM(valor_sinistro), 2) AS custo_total,
  ROUND(AVG(valor_sinistro), 2) AS ticket_medio,
  ROUND(AVG(dias), 0) AS dias_medios
FROM escalacao
GROUP BY especialidade_ps, especialidade_destino, tipo_destino
HAVING COUNT(*) >= 3
ORDER BY custo_total DESC
LIMIT 25

ps_especialidade_origem,destino_especialidade,tipo_evento_destino,qtd_escalacoes,custo_total,ticket_medio,dias_medios
Clínica Geral,Clínica Geral,Cirurgia,4,308163.28,77040.82,55.0
Clínica Geral,Oncologia,Cirurgia,3,301950.74,100650.25,53.0
Pediatria,Clínica Geral,Cirurgia,3,271884.74,90628.25,42.0
Psiquiatria,Endocrinologia,Internação,4,183395.63,45848.91,46.0
Oftalmologia,Clínica Geral,Cirurgia,3,182627.14,60875.71,48.0
Pediatria,Psiquiatria,Internação,4,179013.61,44753.40,18.0
Clínica Geral,Ortopedia,Cirurgia,3,169913.79,56637.93,13.0
Psiquiatria,Clínica Geral,Internação,3,157620.73,52540.24,59.0
Oncologia,Oftalmologia,Internação,3,154615.85,51538.62,65.0
Cardiologia,Clínica Geral,Internação,4,148782.21,37195.55,60.0


## 6. Eficiência de Rede por Operadora — Mesmo Procedimento, Custos Diferentes?

**Pergunta:** Para o mesmo tipo de evento e especialidade, operadoras diferentes têm custos sistematicamente distintos? Isso revelaria diferenças de negociação com a rede credenciada.

**Por que importa:** Se a Operadora A paga 30% mais por cirurgias ortopédicas que a Operadora B, há oportunidade de renegociar ou redirecionar fluxo.

**O que fazer:** Comparar ticket médio por operadora para os top 5 procedimentos; usar como benchmark em negociação de rede; considerar migração de beneficiários para operadoras mais eficientes.

In [0]:
%sql
-- Eficiência de rede: ticket médio por operadora para cada tipo de evento
SELECT
  o.operadora_nome,
  u.tipo_evento,
  COUNT(*) AS qtd_eventos,
  ROUND(AVG(u.valor_sinistro), 2) AS ticket_medio,
  ROUND(MIN(u.valor_sinistro), 2) AS ticket_min,
  ROUND(MAX(u.valor_sinistro), 2) AS ticket_max,
  ROUND(STDDEV(u.valor_sinistro), 2) AS desvio_padrao,
  -- Compara com média geral do tipo
  ROUND(
    AVG(u.valor_sinistro) * 100.0 / AVG(AVG(u.valor_sinistro)) OVER (PARTITION BY u.tipo_evento) - 100, 1
  ) AS pct_vs_media_mercado
FROM homologacao.salutar_saude.gold_fat_utilizacao u
  JOIN homologacao.salutar_saude.gold_dim_plano p ON u.plano_id = p.plano_id
  JOIN homologacao.salutar_saude.gold_dim_operadora o ON p.operadora_id = o.operadora_id
WHERE u.data_evento BETWEEN DATE '2024-07-01' AND DATE '2025-06-30'
  AND u.tipo_evento IN ('Cirurgia', 'Internação')
GROUP BY o.operadora_nome, u.tipo_evento
HAVING COUNT(*) >= 20
ORDER BY u.tipo_evento, ticket_medio DESC

operadora_nome,tipo_evento,qtd_eventos,ticket_medio,ticket_min,ticket_max,desvio_padrao,pct_vs_media_mercado
SulVida Seguros,Cirurgia,80,82159.03,6027.22,147850.51,40773.3,5.9
Boavida Saúde,Cirurgia,72,81971.66,8746.98,148738.44,40323.42,5.7
Unicare Coop,Cirurgia,109,81155.29,5656.74,149698.67,46421.41,4.6
NovaClin Intermédica,Cirurgia,114,78744.24,8146.80,148842.38,43592.44,1.5
Sanare Saúde,Cirurgia,108,75618.51,5155.12,148333.48,41105.34,-2.5
PlenaCare,Cirurgia,70,74848.10,10580.30,147785.20,40344.94,-3.5
Vitamed,Cirurgia,166,74341.76,6745.55,148302.44,41279.07,-4.2
Aurora Saúde,Cirurgia,96,71679.61,5522.51,148691.48,39517.1,-7.6
Sanare Saúde,Internação,89,44875.39,4858.29,79654.56,21314.37,9.1
Unicare Coop,Internação,92,43117.17,4275.31,79129.61,23552.49,4.8


Databricks visualization. Run in Databricks to view.

## 7. Variabilidade Intragrupo — O Verdadeiro Problema de Precificação

**Achado:** Não há outliers estatísticos com Z > 2.5 porque o **desvio padrão dentro de cada grupo (tipo+especialidade) é enorme** — cirurgias variam de R$ 5K a R$ 149K na mesma especialidade. Isso indica que o "mesmo procedimento" tem preços radicalmente diferentes.

**Por que importa:** A ausência de outliers NÃO é boa notícia — significa que a variabilidade é sistêmica. Todos os preços são imprevisíveis, o que torna a precificação de contratos extremamente arriscada.

**O que fazer:** Negociar pacotes fechados por procedimento com a rede credenciada; implementar tabela de referência com teto por tipo+especialidade; auditar eventos acima do P90 de cada grupo.

In [0]:
%sql
-- Variabilidade: amplitude de preços dentro de cada grupo tipo+especialidade
-- Coeficiente de variação (CV) > 50% = alta imprevisibilidade
SELECT
  tipo_evento,
  especialidade,
  COUNT(*) AS qtd_eventos,
  ROUND(AVG(valor_sinistro), 2) AS media,
  ROUND(MIN(valor_sinistro), 2) AS minimo,
  ROUND(PERCENTILE(valor_sinistro, 0.5), 2) AS mediana,
  ROUND(PERCENTILE(valor_sinistro, 0.9), 2) AS p90,
  ROUND(MAX(valor_sinistro), 2) AS maximo,
  ROUND(STDDEV(valor_sinistro), 2) AS desvio_padrao,
  ROUND(STDDEV(valor_sinistro) * 100.0 / AVG(valor_sinistro), 1) AS coef_variacao_pct,
  ROUND(MAX(valor_sinistro) / MIN(valor_sinistro), 1) AS ratio_max_min
FROM homologacao.salutar_saude.gold_fat_utilizacao
WHERE data_evento BETWEEN DATE '2024-07-01' AND DATE '2025-06-30'
  AND tipo_evento IN ('Cirurgia', 'Internação')
GROUP BY tipo_evento, especialidade
HAVING COUNT(*) >= 10
ORDER BY coef_variacao_pct DESC

tipo_evento,especialidade,qtd_eventos,media,minimo,mediana,p90,maximo,desvio_padrao,coef_variacao_pct,ratio_max_min
Internação,Psiquiatria,78,39840.23,3308.73,37661.31,69663.83,79674.16,23671.9,59.4,24.1
Cirurgia,Pediatria,65,65775.88,8973.94,64718.26,119651.45,137899.91,38090.2,57.9,15.4
Internação,Dermatologia,75,39548.63,3296.24,41570.0,71911.84,79129.61,22623.09,57.2,24.0
Cirurgia,Ginecologia,69,77317.22,6745.55,85625.97,133958.23,147801.21,44230.93,57.2,21.9
Cirurgia,Endocrinologia,74,74552.18,6736.76,76422.52,132585.65,148354.76,42125.6,56.5,22.0
Internação,Ortopedia,81,38711.80,3471.18,36738.2,69041.9,79910.08,21783.79,56.3,23.0
Internação,Cardiologia,81,38385.40,3999.93,37222.64,73567.47,79654.56,21544.75,56.1,19.9
Internação,Endocrinologia,83,42331.62,4078.77,44862.63,73573.17,79563.41,23620.84,55.8,19.5
Cirurgia,Oncologia,97,77293.68,5155.12,77400.97,129326.88,148333.48,42998.69,55.6,28.8
Cirurgia,Ortopedia,87,75735.72,9616.98,69988.28,139195.77,149635.33,41769.0,55.2,15.6


In [0]:
%sql
-- Economia potencial: se tivesse teto no P90 de cada grupo, quanto economizaria?
WITH stats AS (
  SELECT tipo_evento, especialidade,
    PERCENTILE(valor_sinistro, 0.9) AS p90
  FROM homologacao.salutar_saude.gold_fat_utilizacao
  WHERE data_evento BETWEEN DATE '2024-07-01' AND DATE '2025-06-30'
  GROUP BY tipo_evento, especialidade
  HAVING COUNT(*) >= 10
),
acima_p90 AS (
  SELECT
    u.evento_id,
    u.tipo_evento,
    u.especialidade,
    u.valor_sinistro,
    s.p90,
    u.valor_sinistro - s.p90 AS excesso_acima_p90
  FROM homologacao.salutar_saude.gold_fat_utilizacao u
    JOIN stats s ON u.tipo_evento = s.tipo_evento AND u.especialidade = s.especialidade
  WHERE u.data_evento BETWEEN DATE '2024-07-01' AND DATE '2025-06-30'
    AND u.valor_sinistro > s.p90
)
SELECT
  COUNT(*) AS eventos_acima_p90,
  ROUND(SUM(excesso_acima_p90), 2) AS economia_potencial_teto_p90,
  ROUND(SUM(excesso_acima_p90) * 100.0 / (SELECT SUM(valor_sinistro) FROM homologacao.salutar_saude.gold_fat_utilizacao WHERE data_evento BETWEEN DATE '2024-07-01' AND DATE '2025-06-30'), 1) AS pct_economia_sobre_total,
  ROUND(AVG(excesso_acima_p90), 2) AS excesso_medio_por_evento
FROM acima_p90

eventos_acima_p90,economia_potencial_teto_p90,pct_economia_sobre_total,excesso_medio_por_evento
547,960008.35,1.0,1755.04


## 8. Síntese dos Achados e Ações Recomendadas

| # | Achado | Impacto | Ação Recomendada | Prioridade |
| --- | --- | --- | --- | --- |
| 1 | **Pico sazonal em Jun/25** — Z-score 2.25, 144 cirurgias (4× a média do início do período) | R$ 10,4M em um único mês | Investigar causa-raiz (demanda reprimida? sazonalidade real?); provisionar reserva técnica 40% maior para jun-jul | 🔴 Alta |
| 2 | **Seleção adversa confirmada** — custo per-capita nos 3 primeiros meses é **57% maior** que entre 6-11 meses (R$ 56K vs R$ 35,5K) | R$ 33M gerados por beneficiários com <3 meses | Reforçar carência para cirurgias eletivas; implementar DPS (Declaração Pessoal de Saúde) com triagem atuária | 🔴 Alta |
| 3 | **Corretores todos saudáveis** (sinistralidade <4%) | Bom sinal de subscripção | Manter modelo; monitorar trimestralmente | 🟢 Baixa |
| 4 | **Carteira diversificada** — HHI = 414 (bem abaixo de 1500) | Top contrato = 9% da receita | Risco controlado; porém top 5 contratos = 33% — manter retenção ativa | 🟡 Média |
| 5 | **29% dos usuários de PS escalam para cirurgia/internação em 90 dias** (custo R$ 11,1M) | Janela média de 48 dias para intervenção | Programa de follow-up pós-PS: telemedicina em 48h, consulta em 7d, gerenciamento de caso | 🔴 Alta |
| 6 | **Operadoras com diferença de até 14pp** em ticket médio (SulVida +6% vs Aurora −8%) | R$ 10K+ de diferença por cirurgia | Benchmarking de rede; renegociar com operadoras mais caras; direcionar fluxo | 🟡 Média |
| 7 | **Variabilidade sistêmica de preços** — mesmo tipo+especialidade varia até 28× (min→max) | Precificação imprevisível; risco atuarial | Implementar tabela de referência com teto no P90; pacotes fechados com rede | 🔴 Alta |

---

### Priorização Executiva

**Ações imediatas (próximos 30 dias):**
1. Programa de follow-up pós-PS (interceptar o funil de escalação)
2. Revisar política de carência/DPS para novos beneficiários
3. Estabelecer teto de referência (P90) para pré-autorização

**Ações de médio prazo (60-90 dias):**
4. Investigar causa da sazonalidade (Jun/25 = pico)
5. Renegociar com operadoras acima do benchmark
6. Monitoramento contínuo de sinistralidade por corretor

## 9. Perguntas de Aprofundamento — O que um Analista Sênior Perguntaria Agora

Com os 7 achados validados, um analista sênior não pararia aqui. Ele cruzaria os insights entre si e pediria respostas que geram **ação imediata do time comercial e atuarial**.

---

### P1 — "A seleção adversa é uniforme ou concentrada em empresas específicas?"

**Raciocínio:** Sabemos que beneficiários com <3 meses custam 57% mais. Mas isso é um problema da carteira toda ou de 3-4 empresas que aderiram com população já doente?

**O que o time faria:**
- Se concentrado em poucas empresas → reajuste técnico pontual + carência específica na renovação
- Se generalizado → política de subscripção precisa mudar para toda a carteira
- Cruzar com `data_inicio_relacionamento` da empresa para ver se empresas novas trazem mais risco

**Impacto:** Direcionar se a solução é cirurgia (caso a caso) ou quimioterapia (política geral)

---

### P2 — "Os beneficiários que escalaram do PS são os mesmos que estão no Top 10% de custo?"

**Raciocínio:** Descobrimos dois fenômenos separados — high-cost claimants (42% do custo) e funil PS→Cirurgia (29% escalam). Se forem as **mesmas pessoas**, um único programa resolve dois problemas.

**O que o time faria:**
- Se overlap > 60% → o case management dos high-cost claimants já intercepta o funil. Priorizar uma única iniciativa
- Se overlap baixo → são dois problemas distintos que requerem duas ações paralelas (gestor de caso vs. follow-up pós-PS)
- Medir: `% dos 132 high-cost que tiveram PS nos 90 dias antes da cirurgia`

**Impacto:** Evitar duplicação de esforço e recursos

---

### P3 — "O pico de Jun/25 foi causado por demanda reprimida de carência ou é sazonalidade real?"

**Raciocínio:** 144 cirurgias em Jun/25 (Z=2.25) vs. 35 em Jul/24. Duas hipóteses:
- **H1 (carência):** Beneficiários que entraram em Jan-Mar/25 cumpriram 180 dias de carência e agendaram cirurgias em Jun/25
- **H2 (sazonalidade):** Meses pré-férias historicamente concentram cirurgias eletivas

**O que o time faria:**
- Se H1 → o pico é previsível (f(data_adesao + 180 dias)). Criar modelo de previsão de sinistro baseado em waves de adesão
- Se H2 → negociar com hospitais pacotes para Jan e Jun (carência de 6m e férias)
- Cruzar: `data_adesao` dos beneficiários que operaram em Jun/25 vs. distribuição geral

**Impacto:** R$ 10,4M concentrados em 1 mês — se previsível, pode ser provisionado e negociado

---

### P4 — "A coparticipação é ineficaz porque o valor é baixo ou porque não incide em cirurgias?"

**Raciocínio:** No documento anterior, vimos que copay reduz apenas 5% do custo. Mas precisamos entender o mecanismo: o problema é que o copay é irrelevante para quem vai operar (ex.: R$ 50 de copay em cirurgia de R$ 80K) ou que ele não existe para eventos de alto custo?

**O que o time faria:**
- Verificar se o plano cobra copay em cirurgias/internações ou só em consultas/exames
- Se não incide em alto custo → redesenhar com copay proporcional (ex.: 10% até teto de R$ 5K)
- Se incide mas é valor fixo baixo → mudar para percentual
- Medir elasticidade: em planos com copay, a taxa de cirurgias eletivas é menor?

**Impacto:** Se copay correto reduzir 10% das cirurgias eletivas = R$ 6,3M/ano

---

### P5 — "Quais empresas têm sinistralidade CRESCENTE? (tendência, não nível)"

**Raciocínio:** A análise anterior mostra quem está em prejuízo HOJE. Mas um analista sênior quer saber quem está **piorando** — empresas com sinistralidade 50% hoje mas trajetória ascendente são mais urgentes que empresas estáveis em 110%.

**O que o time faria:**
- Calcular slope da regressão linear da sinistralidade mensal por empresa (6 meses)
- Priorizar abordagem comercial pelas empresas com **slope positivo + sinistralidade > 70%** (estão indo para prejuízo)
- Empresas em prejuízo mas com slope negativo podem estar se corrigindo naturalmente

**Impacto:** Antecipar problemas ao invés de reagir a eles

---

### P6 — "Existe correlação entre porte da empresa e velocidade de escalação no funil PS→Cirurgia?"

**Raciocínio:** Empresas grandes geralmente têm RH mais estruturado e programas de saúde. Se a taxa de escalação for menor em empresas grandes, valida que programas corporativos funcionam.

**O que o time faria:**
- Se grandes têm menos escalação → replicar boas práticas das grandes para as pequenas/médias (benchmark de programa de saúde)
- Se não há diferença → o problema é assistencial (rede credenciada), não corporativo
- Propor para as PMEs um pacote de "saúde preventiva" como add-on comercial

**Impacto:** Novo produto (saúde preventiva corporativa) que reduz sinistralidade E gera receita

---

### P7 — "O ticket médio das cirurgias está subindo ao longo dos 12 meses (inflação médica)?"

**Raciocínio:** Se o custo médio por cirurgia cresceu 15% no período mas o reajuste foi de 8%, a margem está sendo corroida silenciosamente. Isso explicaria parte do prejuízo.

**O que o time faria:**
- Calcular inflação médica implícita (ticket médio Jun/25 vs. Jul/24, mesmo tipo+especialidade)
- Se > IPCA + 5pp → justificativa técnica para reajuste acima do índice contratual
- Segmentar por operadora: quem está repassando mais inflação?

**Impacto:** Fundamentar pedido de reajuste nos contratos corporativos com dados reais

---

### Recomendações de Próximos Passos

| Prioridade | Ação | Dados Necessários | Responsável Sugerido |
| --- | --- | --- | --- |
| 1 | Validar overlap High-Cost × Funil PS (P2) | gold_fat_utilizacao + dim_beneficiario | Data Analytics |
| 2 | Investigar causa do pico Jun/25 (P3) | data_adesao × data_evento dos cirúrgicos | Atuarial |
| 3 | Segmentar seleção adversa por empresa (P1) | fat_utilizacao + dim_empresa + dim_beneficiario | Comercial + Atuarial |
| 4 | Calcular slope de sinistralidade (P5) | gold_fat_sinistralidade (mensal) | Data Analytics |
| 5 | Redesenhar copay com incidência cirúrgica (P4) | dim_plano + fat_utilizacao por tipo | Produto + Atuarial |
| 6 | Medir inflação médica implícita (P7) | fat_utilizacao mensal por tipo+especialidade | Financeiro |
| 7 | Benchmark de programas por porte (P6) | fat_utilizacao + dim_empresa.porte | Saúde Populacional |